# RAG Chunking Strategies
Chunking is one of the most important — and often underestimated — components of modern RAG and retrieval pipelines. The way a document is divided can significantly affect embedding quality, retrieval accuracy, context preservation, and downstream LLM performance. In this article, we compare eight different chunking strategies: Fixed-size, Recursive, Document-based, Semantic, LLM-based, Agentic, Late, and Hierarchical chunking. Instead of discussing them theoretically, every strategy is executed on the exact same sample document to make the differences directly observable and practically comparable.

To create a realistic stress test, all methods were run on a ~3,000-character technical article about the Transformer architecture containing real section headers, varied paragraph lengths, and natural topic shifts. The outputs shown are actual outputs generated from running the code — not hypothetical examples. Embedding-based approaches use text-embedding-3-small, while LLM-powered strategies use gpt-4o-mini, allowing us to examine how each technique balances simplicity, semantic coherence, structural awareness, and computational cost when processing identical input.

## Setting up the dependencies

In [ ]:
!pip install openai langchain langchain-text-splitters numpy

In [3]:
import os, re, json, textwrap
import numpy as np
from openai import OpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from getpass import getpass

In [4]:
OPENAI_API_KEY = getpass('Enter OpenAI API Key: ')
client = OpenAI(api_key=OPENAI_API_KEY)

Enter OpenAI API Key: ··········


The sample document used throughout this article is a compact technical overview of the Transformer architecture, intentionally designed to stress-test different chunking strategies. It contains real markdown section headers, paragraphs of varying lengths, topic transitions, and both conceptual and mathematical explanations — making it ideal for evaluating how different chunking methods preserve structure, semantics, and contextual continuity. Since every strategy operates on the exact same input document, the differences in chunk boundaries and coherence become much easier to observe and compare.

To keep the experiments consistent, a shared set of utility functions is used across all implementations. These utilities handle formatted chunk visualization, chunk statistics, embedding generation using text-embedding-3-small, and cosine similarity calculations for embedding-based approaches. This ensures that the comparison focuses entirely on the chunking strategy itself rather than differences in preprocessing or evaluation logic.

In [5]:
DOCUMENT = """
# The Transformer Architecture

## Introduction

The Transformer is a deep learning architecture introduced in the paper "Attention Is All You Need" by Vaswani et al. in 2017. It fundamentally changed the field of natural language processing by replacing recurrent and convolutional layers with a mechanism called self-attention. The model achieved state-of-the-art results on machine translation tasks while being significantly faster to train due to its parallelizable design.

Unlike RNNs, which process tokens sequentially, Transformers process all tokens in a sequence simultaneously. This parallel processing makes them highly efficient on modern hardware accelerators like GPUs and TPUs.

## Self-Attention Mechanism

At the core of every Transformer is the self-attention mechanism. For each token in the input sequence, self-attention computes a weighted sum of all other tokens, where the weights represent how much attention that token should pay to every other token.

The mechanism works by projecting each token into three vectors: Query (Q), Key (K), and Value (V). The attention score between two tokens is computed as the dot product of the query of one token with the key of another, scaled by the square root of the key dimension. These scores are then passed through a softmax to produce attention weights.

Multi-head attention extends this by running several attention operations in parallel, each learning to attend to different aspects of the input — syntactic structure, coreference, semantic roles, and so on.

## Positional Encoding

Because self-attention has no inherent sense of order, Transformers inject positional information into the input embeddings using positional encodings. The original paper used fixed sinusoidal encodings of different frequencies.

Modern variants like BERT and GPT use learned positional embeddings instead, where position representations are parameters trained alongside the rest of the model. More recent models such as RoPE (Rotary Position Embedding) and ALiBi encode position through geometric transformations of the attention scores rather than additive embeddings.

## Feed-Forward Layers

After the attention sub-layer, each Transformer block contains a position-wise feed-forward network (FFN). This consists of two linear transformations with a non-linear activation in between — typically ReLU or GELU.

The FFN is applied independently to each token position, meaning it does not mix information across positions. Research suggests that FFN layers act as key-value memories, storing factual associations learned during training. This is in contrast to attention layers, which primarily perform information routing and aggregation.

## Training and Scale

Transformers are trained using gradient descent with the Adam optimizer, minimizing cross-entropy loss over next-token prediction (in the case of language models) or masked token prediction (in the case of models like BERT).

Scaling has proven to be a dominant lever. Empirical scaling laws (Kaplan et al., 2020) show that model performance improves predictably as a power law with increases in parameters, data, and compute. This insight drove the development of very large models such as GPT-3 (175B parameters), PaLM (540B), and beyond.
"""


# ============================================================
# SHARED UTILITIES
# ============================================================

def banner(title):
    width = 62
    print(f"\n{'═' * width}")
    print(f"  {title}")
    print(f"{'═' * width}")

def show_chunks(chunks, label="Chunks"):
    print(f"\n{'─' * 62}")
    print(f"  {label}  →  {len(chunks)} chunks")
    print(f"{'─' * 62}")
    for i, chunk in enumerate(chunks, 1):
        text    = chunk.strip()
        preview = text[:110] + ("..." if len(text) > 110 else "")
        print(f"\n  Chunk {i}  ({len(text)} chars)")
        for line in textwrap.wrap(preview, width=58):
            print(f"    {line}")
    avg = sum(len(c) for c in chunks) // len(chunks)
    print(f"\n  Avg chunk size : {avg} chars")
    print(f"{'─' * 62}")

def get_embeddings(texts, model="text-embedding-3-small"):
    """Batch-embed a list of strings. Returns a 2-D numpy array (N × D)."""
    response = client.embeddings.create(input=texts, model=model)
    return np.array([item.embedding for item in response.data])

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

## Fixed-Size Chunking
Fixed-size chunking is the simplest and most deterministic chunking strategy: the document is split every fixed number of characters (400 in this example) with a small overlap of 50 characters to reduce context loss at boundaries. As seen in the output, this approach produces highly uniform chunk sizes and predictable chunk counts, making it useful for quick experiments and token-budget testing. However, because it ignores document structure and linguistic boundaries, several chunks begin or end mid-sentence or even mid-word (for example, “ry (Q), Key (K)...” and “ute. This insight drove...” ). While overlaps partially mitigate abrupt cuts, the resulting chunks often lack semantic coherence compared to more structure-aware approaches.

In [6]:
def fixed_size_chunking(text, chunk_size=400, overlap=50):
    chunks, start = [], 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        # stride = chunk_size - overlap so the next chunk sees
        # the last `overlap` chars of the current one
        start += chunk_size - overlap
    return chunks


banner("Strategy 1 — Fixed-Size Chunking  (chunk=400, overlap=50)")
fixed_chunks = fixed_size_chunking(DOCUMENT, chunk_size=400, overlap=50)
show_chunks(fixed_chunks, "Fixed-Size")


══════════════════════════════════════════════════════════════
  Strategy 1 — Fixed-Size Chunking  (chunk=400, overlap=50)
══════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────
  Fixed-Size  →  10 chunks
──────────────────────────────────────────────────────────────

  Chunk 1  (399 chars)
    # The Transformer Architecture  ## Introduction  The
    Transformer is a deep learning architecture introduced
    in...

  Chunk 2  (400 chars)
    state-of-the-art results on machine translation tasks
    while being significantly faster to train due to its
    par...

  Chunk 3  (400 chars)
    Self-Attention Mechanism  At the core of every Transformer
    is the self-attention mechanism. For each token in ...

  Chunk 4  (400 chars)
    ry (Q), Key (K), and Value (V). The attention score
    between two tokens is computed as the dot product of the
    q...

  Chunk 5  (400 chars)
    ns in parallel, each learning to at

## Recursive Chunking
Recursive chunking improves significantly over fixed-size chunking by attempting to split text at natural boundaries using a hierarchy of separators such as double newlines, single newlines, periods, and finally spaces. Instead of cutting blindly at a character limit, it recursively searches for the best possible split point below the chunk threshold, resulting in chunks that are far more readable and semantically coherent.

In the output, most chunks align naturally with paragraphs or section transitions like “Self-Attention Mechanism” and “Feed-Forward Layers,” preserving the document’s logical flow. However, because the separator hierarchy is manually defined, the strategy still relies on heuristics rather than actual semantic understanding, and chunk sizes can vary considerably — from very small heading-only chunks to larger explanatory sections. Even so, recursive chunking remains one of the most practical and widely used default strategies due to its simplicity, zero API cost, and strong balance between structure preservation and efficiency.

In [7]:
banner("Strategy 2 — Recursive Chunking  (chunk=400, overlap=50)")

splitter = RecursiveCharacterTextSplitter(
    separators   = ["\n\n", "\n", ".", " "],
    chunk_size   = 400,
    chunk_overlap= 50,
    length_function=len,
)
recursive_chunks = splitter.split_text(DOCUMENT)
show_chunks(recursive_chunks, "Recursive")


══════════════════════════════════════════════════════════════
  Strategy 2 — Recursive Chunking  (chunk=400, overlap=50)
══════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────
  Recursive  →  13 chunks
──────────────────────────────────────────────────────────────

  Chunk 1  (47 chars)
    # The Transformer Architecture  ## Introduction

  Chunk 2  (279 chars)
    The Transformer is a deep learning architecture introduced
    in the paper "Attention Is All You Need" by Vaswani...

  Chunk 3  (150 chars)
    . The model achieved state-of-the-art results on machine
    translation tasks while being significantly faster to...

  Chunk 4  (243 chars)
    Unlike RNNs, which process tokens sequentially,
    Transformers process all tokens in a sequence
    simultaneously. ...

  Chunk 5  (283 chars)
    ## Self-Attention Mechanism  At the core of every
    Transformer is the self-attention mechanism. For each
    toke

## Document-Based Chunking
Document-based chunking leverages the document’s inherent structure instead of relying on arbitrary size limits or separator heuristics. In this example, markdown ## headers are used as chunk boundaries, causing each major Transformer topic — such as “Self-Attention Mechanism” or “Training and Scale” — to become its own self-contained chunk. The resulting chunks are highly interpretable and semantically meaningful, closely matching how humans naturally organize and consume technical content. This makes the strategy particularly effective for documentation systems, wikis, research papers, and manuals where headings accurately represent topic boundaries.

However, the approach depends heavily on the quality and consistency of the source formatting; poorly structured documents or raw text without clear section markers can lead to overly large, uneven, or logically incomplete chunks.

In [8]:
def document_based_chunking(text):
    # re.split with a capturing group keeps the delimiter
    parts = re.split(r'(?m)^(##\s+.+)$', text)
    chunks, i = [], 0
    while i < len(parts):
        part = parts[i].strip()
        if part.startswith("##"):
            body = parts[i + 1].strip() if i + 1 < len(parts) else ""
            chunks.append(f"{part}\n\n{body}")
            i += 2
        else:
            if part:
                chunks.append(part)
            i += 1
    return [c for c in chunks if c.strip()]


banner("Strategy 3 — Document-Based Chunking  (split on ## headers)")
doc_chunks = document_based_chunking(DOCUMENT)
show_chunks(doc_chunks, "Document-Based")


══════════════════════════════════════════════════════════════
  Strategy 3 — Document-Based Chunking  (split on ## headers)
══════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────
  Document-Based  →  6 chunks
──────────────────────────────────────────────────────────────

  Chunk 1  (30 chars)
    # The Transformer Architecture

  Chunk 2  (662 chars)
    ## Introduction  The Transformer is a deep learning
    architecture introduced in the paper "Attention Is All
    You...

  Chunk 3  (839 chars)
    ## Self-Attention Mechanism  At the core of every
    Transformer is the self-attention mechanism. For each
    token ...

  Chunk 4  (594 chars)
    ## Positional Encoding  Because self-attention has no
    inherent sense of order, Transformers inject positional
    ...

  Chunk 5  (569 chars)
    ## Feed-Forward Layers  After the attention sub-layer,
    each Transformer block contains a position-wise feed-fo...


## Semantic Chunking
Semantic chunking moves beyond structural heuristics by using embeddings to detect actual shifts in meaning between consecutive sentences. The process first embeds every sentence, computes cosine similarity between neighboring pairs, and then dynamically determines chunk boundaries using a percentile-based threshold rather than a hardcoded similarity value. This is important because embedding models produce different similarity distributions; in this case, text-embedding-3-small generated similarities mostly between 0.2 and 0.6, making a fixed threshold unreliable. By splitting only at the sharpest semantic drops (bottom 25% here), the strategy adapts naturally to the document and embedding model.

The resulting chunks are generally more semantically coherent than recursive or fixed-size chunking, especially across flowing explanations. However, the method can still produce uneven chunk sizes or overly small chunks when a single sentence appears semantically isolated, as seen with the standalone sinusoidal encoding sentence. Despite this, semantic chunking is highly effective for essays, transcripts, and research-style prose where topic transitions matter more than formatting structure.

In [14]:
def semantic_chunking(text, breakpoint_percentile=25):
    """
    breakpoint_percentile: split at similarity drops in the
    bottom N% of the distribution. Lower = fewer, larger chunks.
    Typical range: 10 (very few splits) to 40 (more splits).
    """
    raw       = re.split(r'(?<=[.?!])\s+', text.strip())
    sentences = [s.strip() for s in raw if len(s.strip()) > 20]

    print(f"\n  Embedding {len(sentences)} sentences...")
    embeddings = get_embeddings(sentences)

    similarities = [
        cosine_similarity(embeddings[i], embeddings[i + 1])
        for i in range(len(embeddings) - 1)
    ]

    # Dynamic threshold: bottom Nth percentile of similarities
    threshold = np.percentile(similarities, breakpoint_percentile)

    print(f"\n  Similarity distribution:")
    print(f"    Min   : {min(similarities):.3f}")
    print(f"    Max   : {max(similarities):.3f}")
    print(f"    Mean  : {np.mean(similarities):.3f}")
    print(f"    Threshold (p{breakpoint_percentile}): {threshold:.3f}  ← split below this")

    print(f"\n  Cosine similarity between consecutive sentences:")
    print(f"  (dynamic threshold = {threshold:.3f}  |  below → chunk boundary)")
    for i, sim in enumerate(similarities):
        marker = "  ← SPLIT" if sim < threshold else ""
        print(f"    [{i:02d}→{i+1:02d}]  {sim:.3f}{marker}")

    chunks, current = [], [sentences[0]]
    for i, sim in enumerate(similarities):
        if sim < threshold:
            chunks.append(" ".join(current))
            current = []
        current.append(sentences[i + 1])
    if current:
        chunks.append(" ".join(current))

    return chunks


banner("Strategy 4 — Semantic Chunking  (dynamic threshold, p25)")
semantic_chunks = semantic_chunking(DOCUMENT, breakpoint_percentile=25)
show_chunks(semantic_chunks, "Semantic")


══════════════════════════════════════════════════════════════
  Strategy 4 — Semantic Chunking  (dynamic threshold, p25)
══════════════════════════════════════════════════════════════

  Embedding 24 sentences...

  Similarity distribution:
    Min   : 0.198
    Max   : 0.602
    Mean  : 0.393
    Threshold (p25): 0.342  ← split below this

  Cosine similarity between consecutive sentences:
  (dynamic threshold = 0.342  |  below → chunk boundary)
    [00→01]  0.479
    [01→02]  0.370
    [02→03]  0.330  ← SPLIT
    [03→04]  0.385
    [04→05]  0.307  ← SPLIT
    [05→06]  0.534
    [06→07]  0.362
    [07→08]  0.410
    [08→09]  0.602
    [09→10]  0.398
    [10→11]  0.420
    [11→12]  0.299  ← SPLIT
    [12→13]  0.254  ← SPLIT
    [13→14]  0.587
    [14→15]  0.354
    [15→16]  0.416
    [16→17]  0.210  ← SPLIT
    [17→18]  0.462
    [18→19]  0.434
    [19→20]  0.198  ← SPLIT
    [20→21]  0.353
    [21→22]  0.437
    [22→23]  0.448

───────────────────────────────────────────────────────

## LLM-Based Chunking
LLM-based chunking uses gpt-4o-mini to directly analyze the document and identify natural discourse boundaries instead of relying on size limits, separators, or embedding similarity scores. Because the model understands topic flow and semantic transitions at a higher level, the resulting chunks are typically the most coherent and human-like among all strategies.

In this example, the model places boundaries at points where complete ideas conclude — such as after the explanation of multi-head attention or positional embedding variants — producing large but semantically unified chunks. Unlike recursive or semantic chunking, the boundaries are driven by contextual understanding rather than heuristics or local sentence similarity.

However, this quality comes at a cost: every document requires an LLM call, making the approach slower and more expensive at scale. For high-value content such as legal documents, contracts, research papers, or enterprise knowledge bases, the improved chunk quality can often justify the additional computational overhead.

In [10]:
def llm_based_chunking(text):
    prompt = f"""You are a document chunking expert. Analyze the following document and identify the best points to split it into coherent, self-contained chunks.

Rules:
- Each chunk should cover one clear idea or topic.
- Return ONLY a JSON array of strings, where each string is the LAST sentence of a chunk (verbatim from the text).
- Do not include the final sentence of the document (it ends the last chunk automatically).
- Aim for 4–6 chunks total.

Document:
\"\"\"{text}\"\"\"

Return ONLY valid JSON. Example format: ["sentence that ends chunk 1.", "sentence that ends chunk 2."]"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    raw = response.choices[0].message.content.strip()
    raw = re.sub(r'^```(?:json)?\s*', '', raw)
    raw = re.sub(r'\s*```$', '', raw)

    boundary_sentences = json.loads(raw)

    print(f"\n  LLM identified {len(boundary_sentences)} chunk boundaries:")
    for i, b in enumerate(boundary_sentences, 1):
        print(f"    [{i}] ...{b[-65:]}")

    chunks, remaining = [], text
    for boundary in boundary_sentences:
        idx = remaining.find(boundary)
        if idx == -1:
            continue
        split_pos = idx + len(boundary)
        chunks.append(remaining[:split_pos].strip())
        remaining = remaining[split_pos:].strip()
    if remaining:
        chunks.append(remaining)

    return chunks


banner("Strategy 5 — LLM-Based Chunking  (GPT-4o-mini boundary detection)")
llm_chunks = llm_based_chunking(DOCUMENT)
show_chunks(llm_chunks, "LLM-Based")


══════════════════════════════════════════════════════════════
  Strategy 5 — LLM-Based Chunking  (GPT-4o-mini boundary detection)
══════════════════════════════════════════════════════════════

  LLM identified 5 chunk boundaries:
    [1] ...g significantly faster to train due to its parallelizable design.
    [2] ...ut — syntactic structure, coreference, semantic roles, and so on.
    [3] ...rmations of the attention scores rather than additive embeddings.
    [4] ...e memories, storing factual associations learned during training.
    [5] ... models such as GPT-3 (175B parameters), PaLM (540B), and beyond.

──────────────────────────────────────────────────────────────
  LLM-Based  →  5 chunks
──────────────────────────────────────────────────────────────

  Chunk 1  (478 chars)
    # The Transformer Architecture  ## Introduction  The
    Transformer is a deep learning architecture introduced
    in...

  Chunk 2  (1055 chars)
    Unlike RNNs, which process tokens sequentially,
   

## Agentic Chunking
Agentic chunking adds a decision-making layer on top of chunk generation by allowing an LLM agent to first inspect the document and determine which chunking strategy is most appropriate before applying it. In this example, the agent correctly identifies the input as a structured technical article and chooses document-based chunking because the markdown sections already represent strong semantic boundaries. One major advantage of this approach is adaptability: instead of hardcoding a single strategy for every input, the system can dynamically switch between semantic, recursive, document-based, or other methods depending on the document type.

The agent also returns its reasoning, making the pipeline more interpretable and easier to debug. However, this flexibility introduces additional latency, cost, and non-determinism since the strategy selection itself requires an extra LLM call. Agentic chunking is particularly useful in real-world ingestion pipelines where document formats vary significantly across PDFs, wikis, research papers, transcripts, and reports.

In [11]:
def agentic_chunking(text):
    system_prompt = """You are a RAG pipeline agent responsible for chunking documents.
Given a document, you will:
1. Identify the document type and structure.
2. Choose the most appropriate chunking strategy from: fixed-size, recursive, document-based, semantic.
3. Apply that strategy and return the result.

Respond ONLY with a JSON object (no markdown, no preamble) in this exact format:
{
  "document_type": "...",
  "chosen_strategy": "...",
  "reasoning": "...",
  "chunks": ["chunk1 text", "chunk2 text", ...]
}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": f"Chunk this document:\n\n{text}"},
        ],
        temperature=0,
    )

    raw = response.choices[0].message.content.strip()
    raw = re.sub(r'^```(?:json)?\s*', '', raw)
    raw = re.sub(r'\s*```$', '', raw)

    result = json.loads(raw)
    print(f"\n  Document type   : {result['document_type']}")
    print(f"  Chosen strategy : {result['chosen_strategy']}")
    print(f"  Reasoning       : {result['reasoning']}")

    return result["chunks"]


banner("Strategy 6 — Agentic Chunking  (agent selects strategy)")
agentic_chunks = agentic_chunking(DOCUMENT)
show_chunks(agentic_chunks, "Agentic")


══════════════════════════════════════════════════════════════
  Strategy 6 — Agentic Chunking  (agent selects strategy)
══════════════════════════════════════════════════════════════

  Document type   : technical article
  Chosen strategy : document-based
  Reasoning       : The document is structured with clear sections and subsections, making it suitable for a document-based chunking strategy that respects these divisions for better comprehension.

──────────────────────────────────────────────────────────────
  Agentic  →  6 chunks
──────────────────────────────────────────────────────────────

  Chunk 1  (30 chars)
    # The Transformer Architecture

  Chunk 2  (662 chars)
    ## Introduction  The Transformer is a deep learning
    architecture introduced in the paper "Attention Is All
    You...

  Chunk 3  (839 chars)
    ## Self-Attention Mechanism  At the core of every
    Transformer is the self-attention mechanism. For each
    token ...

  Chunk 4  (594 chars)
    ## Posi

## Late Chunking
Late chunking addresses one of the major weaknesses of traditional chunking pipelines: isolated embeddings lose surrounding context. In a standard retrieval pipeline, text is first split into chunks and each chunk is embedded independently, meaning the embedding model has no awareness of neighboring content, topic continuity, or cross-sentence references. Late chunking reverses this process by first generating context-aware representations from larger windows of text and only then applying chunk boundaries.

Since the OpenAI embeddings API returns pooled embeddings rather than token-level hidden states, this implementation simulates late chunking by embedding each chunk together with surrounding context while still storing only the target chunk for retrieval. The similarity comparison clearly demonstrates the benefit: the context-aware embedding produces a much stronger relationship between adjacent chunks (0.7863 vs 0.6101), indicating better preservation of semantic continuity.

Although this approach requires additional embedding calls and true late chunking ideally needs access to model internals, it can substantially improve retrieval quality for long-form documents where meaning frequently spans chunk boundaries.

In [12]:
def late_chunking(text, chunk_size=400, context_window=200):
    # Build fixed-size chunk boundaries
    boundaries = []
    start = 0
    while start < len(text):
        boundaries.append((start, min(start + chunk_size, len(text))))
        start += chunk_size

    chunk_texts = [text[s:e] for s, e in boundaries]

    # For each chunk, embed it WITH surrounding context
    context_enriched = []
    for s, e in boundaries:
        ctx_s = max(0, s - context_window)
        ctx_e = min(len(text), e + context_window)
        context_enriched.append(text[ctx_s:ctx_e])

    print(f"\n  Embedding {len(chunk_texts)} context-enriched windows...")
    late_embeddings     = get_embeddings(context_enriched)   # with context
    isolated_embeddings = get_embeddings(chunk_texts[:2])    # without context

    sim_isolated = cosine_similarity(isolated_embeddings[0], isolated_embeddings[1])
    sim_late     = cosine_similarity(late_embeddings[0],     late_embeddings[1])

    print(f"\n  Similarity between chunk 1 & chunk 2:")
    print(f"    Isolated embedding (no context)  : {sim_isolated:.4f}")
    print(f"    Late-chunked embedding (context) : {sim_late:.4f}")
    print(f"    → Context-aware embeddings capture cross-chunk")
    print(f"      relationships that isolated embeddings miss.")

    return list(zip(chunk_texts, late_embeddings))


banner("Strategy 7 — Late Chunking  (embed-first, chunk-later)")
late_result  = late_chunking(DOCUMENT, chunk_size=400, context_window=200)
late_texts   = [c for c, _ in late_result]
show_chunks(late_texts, "Late Chunking (text only; embeddings are context-aware)")


══════════════════════════════════════════════════════════════
  Strategy 7 — Late Chunking  (embed-first, chunk-later)
══════════════════════════════════════════════════════════════

  Embedding 9 context-enriched windows...

  Similarity between chunk 1 & chunk 2:
    Isolated embedding (no context)  : 0.6101
    Late-chunked embedding (context) : 0.7863
    → Context-aware embeddings capture cross-chunk
      relationships that isolated embeddings miss.

──────────────────────────────────────────────────────────────
  Late Chunking (text only; embeddings are context-aware)  →  9 chunks
──────────────────────────────────────────────────────────────

  Chunk 1  (399 chars)
    # The Transformer Architecture  ## Introduction  The
    Transformer is a deep learning architecture introduced
    in...

  Chunk 2  (400 chars)
    sks while being significantly faster to train due to its
    parallelizable design.  Unlike RNNs, which process to...

  Chunk 3  (399 chars)
    token in the inp

## Hierarchical Chunking
Hierarchical chunking combines multiple levels of granularity to balance retrieval precision with contextual completeness. Instead of producing only one set of chunks, the document is split into large parent chunks (entire sections) and smaller child chunks (individual paragraphs within those sections), with explicit parent-child relationships maintained between them. This enables a powerful retrieval workflow: the system can first retrieve highly specific child paragraphs for accurate semantic matching, then provide the corresponding parent section to the LLM so it receives the broader context needed for high-quality generation.

In the Transformer article example, each major section becomes a parent chunk while the explanatory paragraphs inside become children, creating a structured representation of the document rather than a flat list of chunks. Although this approach increases storage requirements because both parent and child embeddings must be maintained, it is highly effective for long and information-dense content such as textbooks, technical specifications, enterprise documentation, and research reports where fine-grained retrieval alone may lose important surrounding context.

In [13]:
def hierarchical_chunking(text):
    raw_sections = re.split(r'(?m)^(##\s+.+)$', text)
    sections, i, sec_id = [], 0, 0

    while i < len(raw_sections):
        part = raw_sections[i].strip()
        if part.startswith("##"):
            title = part.lstrip("#").strip()
            body  = raw_sections[i + 1].strip() if i + 1 < len(raw_sections) else ""
            paras = [p.strip() for p in body.split("\n\n") if p.strip()]
            sections.append({
                "section_id":    sec_id,
                "section_title": title,
                "section_text":  body,
                "paragraphs":    paras,
            })
            sec_id += 1
            i += 2
        else:
            i += 1
    return sections


banner("Strategy 8 — Hierarchical Chunking  (section → paragraph)")
hier = hierarchical_chunking(DOCUMENT)

print(f"\n  {len(hier)} sections found\n")
for sec in hier:
    print(f"  ▶ [Section {sec['section_id']}]  {sec['section_title']}")
    print(f"    Parent chunk : {len(sec['section_text'])} chars")
    print(f"    Child chunks : {len(sec['paragraphs'])} paragraphs")
    for j, para in enumerate(sec['paragraphs'], 1):
        preview = para[:80] + ("..." if len(para) > 80 else "")
        print(f"      [{j}] ({len(para)} chars)  {preview}")
    print()

print("  Retrieval pattern:")
print("    match child paragraphs  →  return parent section to LLM")

hier_all_paragraphs = [p for sec in hier for p in sec["paragraphs"]]


══════════════════════════════════════════════════════════════
  Strategy 8 — Hierarchical Chunking  (section → paragraph)
══════════════════════════════════════════════════════════════

  5 sections found

  ▶ [Section 0]  Introduction
    Parent chunk : 645 chars
    Child chunks : 2 paragraphs
      [1] (429 chars)  The Transformer is a deep learning architecture introduced in the paper "Attenti...
      [2] (214 chars)  Unlike RNNs, which process tokens sequentially, Transformers process all tokens ...

  ▶ [Section 1]  Self-Attention Mechanism
    Parent chunk : 810 chars
    Child chunks : 3 paragraphs
      [1] (254 chars)  At the core of every Transformer is the self-attention mechanism. For each token...
      [2] (345 chars)  The mechanism works by projecting each token into three vectors: Query (Q), Key ...
      [3] (207 chars)  Multi-head attention extends this by running several attention operations in par...

  ▶ [Section 2]  Positional Encoding
    Parent chunk : 570 c

## Summary Comparison
There is no universally “best” chunking strategy — the right choice depends entirely on the document type, retrieval goals, latency constraints, and infrastructure budget. Simpler approaches like fixed-size and recursive chunking remain highly practical for fast, low-cost pipelines, while document-based and hierarchical chunking excel when strong document structure already exists.

Embedding-driven and LLM-powered approaches introduce significantly better semantic awareness, producing more coherent chunks and stronger retrieval quality at the cost of additional API usage and latency. As the summary demonstrates, different strategies naturally produce very different chunk counts, chunk sizes, and contextual behaviors even when applied to the exact same document.

In practice, modern RAG systems increasingly move toward adaptive and context-aware techniques such as semantic, late, hierarchical, and agentic chunking because retrieval quality is often bottlenecked not by the embedding model itself, but by how information is segmented before retrieval even begins. The key takeaway is that chunking should be treated as a core retrieval design decision rather than a preprocessing afterthought.

In [15]:
banner("Summary — All 8 Strategies")

strategies = [
    ("Fixed-size",     fixed_chunks,          "No",  "Speed / quick prototypes"),
    ("Recursive",      recursive_chunks,      "No",  "General-purpose prose"),
    ("Document-based", doc_chunks,            "No",  "Structured docs (wikis, manuals)"),
    ("Semantic",       semantic_chunks,       "Yes", "Dense prose, topic shifts"),
    ("LLM-based",      llm_chunks,            "Yes", "High-value, complex documents"),
    ("Agentic",        agentic_chunks,        "Yes", "Mixed / unknown document types"),
    ("Late chunking",  late_texts,            "Yes", "Context-sensitive retrieval"),
    ("Hierarchical",   hier_all_paragraphs,   "No",  "Long docs, multi-level retrieval"),
]

col = "{:<18} {:>7}  {:>11}  {:>5}  {}"
print(f"\n  {col.format('Strategy', 'Chunks', 'Avg (chars)', 'API?', 'Best for')}")
print(f"  {'─' * 72}")

for name, chunks, api, use_case in strategies:
    n   = len(chunks)
    avg = sum(len(c) for c in chunks) // n if n else 0
    print(f"  {col.format(name, n, avg, api, use_case)}")



══════════════════════════════════════════════════════════════
  Summary — All 8 Strategies
══════════════════════════════════════════════════════════════

  Strategy            Chunks  Avg (chars)   API?  Best for
  ────────────────────────────────────────────────────────────────────────
  Fixed-size              10          371     No  Speed / quick prototypes
  Recursive               13          257     No  General-purpose prose
  Document-based           6          542     No  Structured docs (wikis, manuals)
  Semantic                 7          463    Yes  Dense prose, topic shifts
  LLM-based                5          652    Yes  High-value, complex documents
  Agentic                  6          542    Yes  Mixed / unknown document types
  Late chunking            9          363    Yes  Context-sensitive retrieval
  Hierarchical            11          281     No  Long docs, multi-level retrieval
